##  Experiments

In [1]:
import json
import numpy as np
import pandas as pd
import os
from google.cloud import bigquery
from google.api_core.exceptions import NotFound
from google.cloud import storage
import joblib
from io import BytesIO
from dotenv import load_dotenv
import types
import sys

from datetime import datetime
import pytz



pd.set_option("display.max_columns", None)

D:\03_PROYECTOS\Project-GCP-Vertex-AI-Pipelines\project-data-processing\.venv\lib\site-packages\google\api_core\_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.9.25). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
D:\03_PROYECTOS\Project-GCP-Vertex-AI-Pipelines\project-data-processing\.venv\lib\site-packages\google\auth\__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
D:\03_PROYECTOS\Project-GCP-Vertex-AI-Pipelines\project-data-processing\.venv\lib\site-packages\google\oauth2\_

In [2]:
load_dotenv()

project_id = os.getenv("GCP_PROJECT_ID")
bucket_name = os.getenv("GCP_BUCKET_NAME")
model_ruta = os.getenv("MODEL_ROOT")
pipeline_ruta = os.getenv("PIPELINE_ROOT")
credentials_path = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

# Tablas inputs
project_id_input = os.getenv("GCP_PROJECT_ID_INPUT")
dataset_id_input = os.getenv("BQ_DATASET_ID_INPUT")
table_id_input = os.getenv("BQ_TABLE_ID_INPUT")
dataset_temp = os.getenv("BQ_DATASET_ID_TEMP")
table_temp_data_input = os.getenv("BQ_TABLE_ID_TEMP_DATA")
table_temp_data_transf_input = os.getenv("BQ_TABLE_ID_TEMP_DATA_TRANSF")

# Tablas features
dataset_id_features = os.getenv("BQ_DATASET_ID_FEATURES")
table_id_features = os.getenv("BQ_TABLE_ID_FEATURES")

# Tablas Ouputs
project_id_out = os.getenv("GCP_PROJECT_ID_OUT")
dataset_id_out = os.getenv("BQ_DATASET_ID_OUT")
table_out = os.getenv("BQ_TABLE_ID_OUT")
table_out_hist = os.getenv("BQ_TABLE_ID_OUT_HIST")


In [3]:
model_name  = "Model-GradientBoostingRegressor.joblib"
model_path = f"{model_ruta}/{model_name}"
print(model_path)

pipeline_name = 'Pipeline-Transformacion-Training.joblib'
pipeline_path = f"{pipeline_ruta}/{pipeline_name}"
print(pipeline_path)

metadata_name = 'metadata_transformer.py'
metadata_path = f"{pipeline_ruta}/{metadata_name}"
print(metadata_path)

project-pipeline-predictions-casas/models/Model-GradientBoostingRegressor.joblib
project-pipeline-predictions-casas/pipeline/Pipeline-Transformacion-Training.joblib
project-pipeline-predictions-casas/pipeline/metadata_transformer.py


In [4]:
# Cliente de Cloud Storage
client = storage.Client(project=project_id)

bucket = client.bucket(bucket_name)
blob = bucket.blob(model_path)

contenido_joblib  = blob.download_as_bytes()

modelo = joblib.load(BytesIO(contenido_joblib))

print("Archivo JOBLIB cargado correctamente ✅")
print(type(modelo))

Archivo JOBLIB cargado correctamente ✅
<class 'sklearn.ensemble._gb.GradientBoostingRegressor'>


D:\03_PROYECTOS\Project-GCP-Vertex-AI-Pipelines\project-data-processing\.venv\lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator DummyRegressor from version 1.9.0 when using version 1.4.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
D:\03_PROYECTOS\Project-GCP-Vertex-AI-Pipelines\project-data-processing\.venv\lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.9.0 when using version 1.4.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
D:\03_PROYECTOS\Project-GCP-Vertex-AI-Pipelines\project-data-processing\.venv\lib\site-packages

In [5]:
blob = bucket.blob(metadata_path)

module_code  = blob.download_as_text(encoding="utf-8")


metadata_transformer = types.ModuleType('metadata_transformer')
metadata_transformer.__file__ = "gs://.../metadata_transformer.py"
sys.modules['metadata_transformer'] = metadata_transformer

exec(compile(module_code, metadata_transformer.__file__, "exec"),
     metadata_transformer.__dict__)

In [6]:
blob = bucket.blob(pipeline_path)

contenido_joblib  = blob.download_as_bytes()

pipeline = joblib.load(BytesIO(contenido_joblib))

print("Archivo JOBLIB cargado correctamente ✅")
print(type(pipeline))

Archivo JOBLIB cargado correctamente ✅
<class 'sklearn.pipeline.Pipeline'>


D:\03_PROYECTOS\Project-GCP-Vertex-AI-Pipelines\project-data-processing\.venv\lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.9.0 when using version 1.4.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
D:\03_PROYECTOS\Project-GCP-Vertex-AI-Pipelines\project-data-processing\.venv\lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.9.0 when using version 1.4.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [7]:
client = bigquery.Client(project=project_id)

path_bq_table = f"{project_id_input}.{dataset_id_input}.{table_id_input}"

dfInput = client.query(
        f'''SELECT * FROM `{path_bq_table}`
        '''
    ).to_dataframe()

path_bq_table_feature = f"{project_id}.{dataset_id_features}.{table_id_features}"
dfFeatures = client.query(
        f'''SELECT * FROM `{path_bq_table_feature}`
        '''
    ).to_dataframe()

listFeatures = dfFeatures['features'].tolist()

dfInputFeat = dfInput[listFeatures].copy()
dfInputFeat['id'] = dfInputFeat.index


ZPeru = pytz.timezone("America/Lima")
fecha_carga = datetime.now(ZPeru)
dfInputFeat['periodo'] = fecha_carga.replace(tzinfo=None).strftime("%Y%m")
dfInputFeat['date_subida_local'] = fecha_carga.replace(tzinfo=None)
dfInputFeat['date_subida_utc'] = fecha_carga.astimezone(pytz.UTC)

dfInputFeat = dfInputFeat[['id','periodo'] + listFeatures + ['date_subida_local', 'date_subida_utc']]
periodo = dfInputFeat['periodo'].iloc[0]
dfInputFeat

D:\03_PROYECTOS\Project-GCP-Vertex-AI-Pipelines\project-data-processing\.venv\lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,id,periodo,mssubclass,mszoning,lotfrontage,lotarea,neighborhood,overallqual,overallcond,yearbuilt,yearremodadd,bsmtqual,bsmtfintype_principal,bsmtfinsf_principal,totalbsmtsf,centralair,firstflrsf,secondflrsf,grlivarea,kitchenqual,fireplacequ,garagetype,garagecars,garagearea,yrsold,date_subida_local,date_subida_utc
0,0,202608,20,RH,80.0,11622,NAmes,5,6,1961,1961,TA,Rec,468.0,882.0,Y,896.0,0.0,896.0,TA,None,Attchd,1.0,730.0,2010,2026-08-06 16:00:40.446526,2026-08-06 21:00:40.446526+00:00
1,1,202608,20,RL,81.0,14267,NAmes,6,6,1958,1958,TA,ALQ,923.0,1329.0,Y,1329.0,0.0,1329.0,Gd,None,Attchd,1.0,312.0,2010,2026-08-06 16:00:40.446526,2026-08-06 21:00:40.446526+00:00
2,2,202608,60,RL,74.0,13830,Gilbert,5,5,1997,1998,Gd,GLQ,791.0,928.0,Y,928.0,701.0,1629.0,TA,TA,Attchd,2.0,482.0,2010,2026-08-06 16:00:40.446526,2026-08-06 21:00:40.446526+00:00
3,3,202608,60,RL,78.0,9978,Gilbert,6,6,1998,1998,TA,GLQ,602.0,926.0,Y,926.0,678.0,1604.0,Gd,Gd,Attchd,2.0,470.0,2010,2026-08-06 16:00:40.446526,2026-08-06 21:00:40.446526+00:00
4,4,202608,120,RL,43.0,5005,StoneBr,8,5,1992,1992,Gd,ALQ,263.0,1280.0,Y,1280.0,0.0,1280.0,Gd,None,Attchd,2.0,506.0,2010,2026-08-06 16:00:40.446526,2026-08-06 21:00:40.446526+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1454,1454,202608,160,RM,21.0,1936,MeadowV,4,7,1970,1970,TA,Unf,0.0,546.0,Y,546.0,546.0,1092.0,TA,None,None,0.0,0.0,2006,2026-08-06 16:00:40.446526,2026-08-06 21:00:40.446526+00:00
1455,1455,202608,160,RM,21.0,1894,MeadowV,4,5,1970,1970,TA,Rec,252.0,546.0,Y,546.0,546.0,1092.0,TA,None,CarPort,1.0,286.0,2006,2026-08-06 16:00:40.446526,2026-08-06 21:00:40.446526+00:00
1456,1456,202608,20,RL,160.0,20000,Mitchel,5,7,1960,1996,TA,ALQ,1224.0,1224.0,Y,1224.0,0.0,1224.0,TA,TA,Detchd,2.0,576.0,2006,2026-08-06 16:00:40.446526,2026-08-06 21:00:40.446526+00:00
1457,1457,202608,85,RL,62.0,10441,Mitchel,5,5,1992,1992,Gd,GLQ,337.0,912.0,Y,970.0,0.0,970.0,TA,None,None,0.0,0.0,2006,2026-08-06 16:00:40.446526,2026-08-06 21:00:40.446526+00:00


In [8]:
path_bq_data_input = f"{project_id}.{dataset_temp}.{table_temp_data_input}"

try:
    client.get_table(path_bq_data_input)

    delete_query = f"""
        DELETE FROM `{path_bq_data_input}`
        WHERE periodo = @periodo
    """

    delete_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("periodo", "STRING", periodo)
        ]
    )

    delete_job = client.query(delete_query, job_config=delete_config)
    delete_job.result()

    print(f"Registros previos eliminados para período {periodo}.")

except NotFound:
    print("La tabla destino no existe aún; se creará durante la carga.")


job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
)

job = client.load_table_from_dataframe(
    dfInputFeat,
    path_bq_data_input,
    job_config=job_config,
)

job.result()

print(f"Tabla cargada: {path_bq_data_input}")
print(f"Filas insertadas: {job.output_rows}")
print(f"Filas que deben insertarse: {dfInputFeat.shape[0]}")

Registros previos eliminados para período 202608.
Tabla cargada: gcp-processing-vertex-prod-us.dev_table.temp_data_input
Filas insertadas: 1459
Filas que deben insertarse: 1459


In [9]:
path_bq_data_transf_input = f"{project_id}.{dataset_temp}.{table_temp_data_transf_input}"

X_escalado = pipeline.transform(dfInputFeat)

feature_order = pipeline.named_steps[
    "feature_engineering"
].get_feature_names_out()

dfInputFeatTransf = pd.DataFrame(
    X_escalado,
    columns = [col + '_transf' for col in feature_order],
    index=  dfInputFeat.index,
)
dfInputFeatTransf['saleprice'] = modelo.predict(dfInputFeatTransf)
dfInputFeatTransf['id'] = dfInputFeat.index


dfOutput = dfInputFeat.drop(columns =['date_subida_local','date_subida_utc']).merge(
    dfInputFeatTransf, on = ['id'], how='left')

ZPeru = pytz.timezone("America/Lima")
fecha_carga = datetime.now(ZPeru)
dfOutput['date_subida_local'] = fecha_carga.replace(tzinfo=None)
dfOutput['date_subida_utc'] = fecha_carga.astimezone(pytz.UTC)

feature_out = [['id','periodo'] + 
    listFeatures +
    [col + '_transf' for col in listFeatures if col not in ('yrsold')] + 
    ['saleprice','date_subida_local','date_subida_utc']
]


try:
    client.get_table(path_bq_data_transf_input)

    delete_query = f"""
        DELETE FROM `{path_bq_data_transf_input}`
        WHERE periodo = @periodo
    """

    delete_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("periodo", "STRING", periodo)
        ]
    )

    delete_job = client.query(delete_query, job_config=delete_config)
    delete_job.result()

    print(f"Registros previos eliminados para período {periodo}.")

except NotFound:
    print("La tabla destino no existe aún; se creará durante la carga.")


job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
)

job = client.load_table_from_dataframe(
    dfOutput,
    path_bq_data_transf_input,
    job_config=job_config,
)

job.result()

print(f"Tabla cargada: {path_bq_data_transf_input}")
print(f"Filas insertadas: {job.output_rows}")
print(f"Filas que deben insertarse: {dfOutput.shape[0]}")

D:\03_PROYECTOS\Project-GCP-Vertex-AI-Pipelines\project-data-processing\.venv\lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but GradientBoostingRegressor was fitted without feature names
  warnings.warn(


Registros previos eliminados para período 202608.
Tabla cargada: gcp-processing-vertex-prod-us.dev_table.temp_data_output
Filas insertadas: 1459
Filas que deben insertarse: 1459


In [12]:
table_id_auditoria = (
    f"{project_id}.{dataset_id_features}.model_execution_audit"
)

# Fecha/hora de la ejecución actual, en UTC
fecha_actualizacion = datetime.now(ZPeru)

df_auditoria = pd.DataFrame([{
    "periodo": periodo,
    "model_name": model_name,
    "model_path_gcs": f"gs://{bucket_name}/{model_path}",
    "pipeline_path_gcs": (
        f"gs://{bucket_name}/{model_ruta}/Pipeline-Transformacion-Training.joblib"
    ),
    "rows_processed": len(dfOutput),
    "updated_at": fecha_actualizacion,
}])

job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
)

job = client.load_table_from_dataframe(
    df_auditoria,
    table_id_auditoria,
    job_config=job_config,
)

job.result()

print(f"Auditoría registrada en: {table_id_auditoria}")

Auditoría registrada en: gcp-processing-vertex-prod-us.dev_table.model_execution_audit


In [19]:
listOut = ['id','periodo'] + listFeatures + ['saleprice','date_subida_local','date_subida_utc']
dfOut = dfOutput[listOut].copy()

table_id_out = f'{project_id_out}.{dataset_id_out}.{table_out}'

job_config_truncate = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

job = client.load_table_from_dataframe(
    dfOut[listOut],
    table_id_out,
    job_config=job_config_truncate,
)

job.result()

print(f"Subida a PRD registrada en: {table_id_out}")

Subida a PRD registrada en: gcp-output-bigquery-prod-us-ea.prod_table.ba_modelo_propension_inscripcion


In [20]:
table_id_out_hist = f'{project_id_out}.{dataset_id_out}.{table_out_hist}'

job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
)

job = client.load_table_from_dataframe(
    dfOut,
    table_id_out_hist,
    job_config=job_config,
)

job.result()

print(f"Subida a PRD registrada en: {table_id_out_hist}")

Subida a PRD registrada en: gcp-output-bigquery-prod-us-ea.prod_table.ba_aux_modelo_universo
